# People Deduper — Kaggle GPU Runner (SEPARATE account)

Finds duplicate actors/crew in the `people` table and **safely merges** them with Qwen3-VL-8B.
Run this on a **different Kaggle account** than the cast enricher so the two never compete.

**Safety:** a duplicate is only deleted after all its credits are re-pointed to the primary.
Credits are never dropped except true duplicates (primary already has the identical credit).
It is **DRY RUN by default** — you review the planned merges first, then set `DEDUPE_DRY_RUN=0`.

### Before you run (right panel):
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**
3. **Add-ons → Secrets:** `VITE_SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`, `GITHUB_TOKEN`

In [ ]:
# 1. System deps + Ollama
import subprocess, time, os, urllib.request
subprocess.run('apt-get -qq update && apt-get -qq install -y zstd', shell=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
print('ok')

In [ ]:
# 2. Start Ollama and pull the 8B text/vision model
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
_log = open('/tmp/ollama.log', 'w')
subprocess.Popen(['ollama', 'serve'], stdout=_log, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434', timeout=2); break
    except Exception:
        time.sleep(2)
subprocess.run(['ollama', 'pull', 'qwen3-vl:8b-instruct'], check=True)
print('model ready')

In [ ]:
# 3. Python deps
!pip -q install rapidfuzz requests

In [ ]:
# 4. Secrets + clone
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
for k in ['VITE_SUPABASE_URL', 'SUPABASE_SERVICE_ROLE_KEY']:
    os.environ[k] = sec.get_secret(k)
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'
os.environ['OLLAMA_TEXT_MODEL'] = 'qwen3-vl:8b-instruct'
token = sec.get_secret('GITHUB_TOKEN') + '@'
subprocess.run(['rm', '-rf', 'muvidb'])
subprocess.run(['git', 'clone', '--depth', '1', '-b', 'staging',
                f'https://{token}github.com/Amichael10/muvidb.git'], check=True)
print('cloned')

In [ ]:
# 5. DRY RUN — review the planned merges. Writes NOTHING.
import sys
env = os.environ.copy()
env['DEDUPE_DRY_RUN'] = '1'
p = subprocess.Popen([sys.executable, '-u', 'people_deduper.py'], cwd='muvidb', env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()

In [ ]:
# 6. APPLY — actually merge. Only run after the dry run looks right.
#    Set DEDUPE_LOOP=1 for a continuous 24/7 pass.
import sys
env = os.environ.copy()
env['DEDUPE_DRY_RUN'] = '0'
env['DEDUPE_LOOP'] = '0'   # set '1' for 24/7
p = subprocess.Popen([sys.executable, '-u', 'people_deduper.py'], cwd='muvidb', env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()